# PubMed Evidence Assistant - Notebook 1 (no API key): Baseline RAG + Evaluation

A RAG assistant that answers biomedical research questions (**yes / no / maybe**) from PubMed abstracts and **cites the abstract it used**.
Built on **PubMedQA** (Hugging Face: `qiaojin/PubMedQA`, MIT licence): 1,000 expert-labelled questions, each tied to its abstract, plus a large unlabeled pool used as distractors.

**No API key and no login.** The LLM (`Qwen/Qwen3-4B-Instruct-2507`, an open model) runs on the Colab T4 through Hugging Face `transformers`. Self-hosting a small model is also a genuine cost/latency story for your README.

**What it measures**
| Question | How |
|---|---|
| Does retrieval find the right abstract among thousands? | hit@k and MRR (gold = the question's own PubMed ID) |
| Is the answer right? | accuracy and macro-F1 against expert labels (no LLM judge needed) |
| Does RAG actually help? | `closed_book` vs `rag` vs `oracle` (oracle = gold abstract handed over, the upper bound) |
| Does it cite correctly? | share of answers that cite the gold abstract |
| What does it cost? | p50/p95 latency, tokens, and an estimated GPU cost per query |

**Stack:** Hugging Face `datasets`, `sentence-transformers` (bge embeddings) and `transformers` (local LLM); LangChain (`Document`, `RecursiveCharacterTextSplitter`); FAISS + BM25 for retrieval.

**Colab setup (once):** Runtime -> Change runtime type -> **T4 GPU**. That's it. Work and caches are saved to `MyDrive/pubmed_rag/`.

**Responsible-AI note (put this in your README):** this tool summarises published research. It is not a medical advice or diagnosis tool, and the system prompt says so.

In [ ]:
!pip -q install datasets sentence-transformers rank_bm25 faiss-cpu langchain-core langchain-text-splitters scikit-learn accelerate bitsandbytes

In [ ]:
import os, re, json, time, hashlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path("/content/drive/MyDrive/pubmed_rag") if IN_COLAB else Path("./pubmed_rag")
if IN_COLAB:
    drive.mount("/content/drive")
RESULTS_DIR, CACHE_DIR = ROOT / "results", ROOT / "cache"
for d in (RESULTS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- local LLM (no API key, no login) ----------
LLM_ID    = "Qwen/Qwen3-4B-Instruct-2507"   # open, non-thinking, ~8 GB in fp16: fits a free T4
LOAD_4BIT = False                           # True -> 4-bit (~3 GB, slightly lower quality). Use it if you run out of GPU memory
WITH_REASON = False                         # True adds a one-sentence reason (nicer demo, slower evals)
MAX_NEW = 120 if WITH_REASON else 40        # decision + sources is only ~20 tokens
GPU_USD_PER_HOUR = 0.0                      # Colab is free. Enter a cloud GPU's hourly price to estimate cost per query

# ---------- retrieval ----------
EMBED_MODEL  = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge-en-v1.5 query instruction (not used for passages)
N_DISTRACTORS = 4000      # extra abstracts that are NOT the answer. Raise to 20000 for a harder, more realistic search.
K = 5                     # top-k chunks sent to the LLM / scored for retrieval

# ---------- evaluation splits ----------
N_DEV, N_TEST = 150, 150  # stratified by label. Tune on DEV. Touch TEST once, at the end.
SEED = 42

## 1. Data

`pqa_labeled` has 1,000 questions with expert yes/no/maybe labels. The abstract the question came from is the gold document.
**Important:** PubMedQA's `context` is the abstract *without* its conclusion; the conclusion is stored separately as `long_answer`. We keep `long_answer` for error analysis only and **never put it in a prompt** (that would leak the answer).

The retrieval pool = the 1,000 gold abstracts + `N_DISTRACTORS` abstracts from `pqa_unlabeled`.
Note: the official PubMedQA leaderboard uses a specific 500-question test set from the authors' GitHub. We use our own stratified dev/test split, so don't compare our numbers directly to the leaderboard.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def to_abstract(ctx):
    """PubMedQA 'context' = parallel lists: contexts (text), labels (section names), meshes."""
    parts = []
    for label, text in zip(ctx["labels"], ctx["contexts"]):
        parts.append(f"{str(label).title()}: {text}" if label else text)
    return "\n".join(parts)

labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
questions = pd.DataFrame([{
    "pubid": str(r["pubid"]), "question": r["question"], "decision": r["final_decision"],
    "abstract": to_abstract(r["context"]),
    "long_answer": r["long_answer"],            # error analysis ONLY - never send to the LLM
} for r in labeled])

pool = dict(zip(questions.pubid, questions.abstract))
extra = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train", streaming=True).take(N_DISTRACTORS)
for r in extra:
    pool.setdefault(str(r["pubid"]), to_abstract(r["context"]))
docs = [{"doc_id": k, "text": v} for k, v in pool.items()]

# fixed, saved splits so results are reproducible
dev_idx, rest_idx = train_test_split(questions.index, train_size=N_DEV, stratify=questions.decision, random_state=SEED)
test_idx, _ = train_test_split(rest_idx, train_size=N_TEST, stratify=questions.loc[rest_idx, "decision"], random_state=SEED)
dev_df, test_df = questions.loc[dev_idx], questions.loc[test_idx]
pd.concat([dev_df.assign(split="dev"), test_df.assign(split="test")])[["pubid", "split"]].to_csv(RESULTS_DIR / "splits.csv", index=False)

print(f"{len(questions)} labelled questions | {len(docs)} abstracts in the pool | avg {np.mean([len(d['text']) for d in docs]):.0f} chars")
print("label share:", questions.decision.value_counts(normalize=True).round(3).to_dict())
print(f"majority-class (always 'yes') accuracy on all labelled: {(questions.decision == 'yes').mean():.3f}  <- the number to beat")

## 2. Chunking (LangChain splitters)

Three strategies to compare: whole abstract (no splitting), and `RecursiveCharacterTextSplitter` at 500 and 1000 characters. The retrieval metric is **document-level** (does a top-k chunk come from the gold abstract?), so it stays comparable across chunkers.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_whole(docs):
    return [{"doc_id": d["doc_id"], "text": d["text"]} for d in docs]

def make_rcts(chunk_chars, overlap):
    def _chunk(docs):
        splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_chars, chunk_overlap=overlap)
        lc_docs = [Document(page_content=d["text"], metadata={"doc_id": d["doc_id"]}) for d in docs]
        return [{"doc_id": c.metadata["doc_id"], "text": c.page_content} for c in splitter.split_documents(lc_docs)]
    return _chunk

CHUNKERS = {"whole_abstract": chunk_whole, "rcts_500": make_rcts(500, 100), "rcts_1000": make_rcts(1000, 200)}
for name, fn in CHUNKERS.items():
    print(f"{name:15s} -> {len(fn(docs[:500]))} chunks from 500 docs")

## 3. Retrieval: BM25, dense (bge), hybrid (RRF)

Embeddings are cached on Drive, so a Colab reset does not cost you the re-encoding time.

In [ ]:
import faiss, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 512
if torch.cuda.is_available():
    embedder.half()                                  # fp16 on the T4: ~2x faster, same quality for retrieval

def tok(s):
    return re.findall(r"\w+", s.lower())

def embed_cached(texts, tag):
    key = hashlib.md5((EMBED_MODEL + "||" + "\x1f".join(texts)).encode("utf-8")).hexdigest()[:16]
    path = CACHE_DIR / f"emb_{tag}_{key}.npy"
    if path.exists():
        return np.load(path)
    emb = np.asarray(embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True), dtype="float32")
    np.save(path, emb)
    return emb

# query embeddings, computed once and reused by every retriever / mode
Q_EMB = np.asarray(embedder.encode([QUERY_PREFIX + q for q in questions.question], normalize_embeddings=True,
                                   batch_size=64, show_progress_bar=False), dtype="float32")
QIDX = {pid: i for i, pid in enumerate(questions.pubid)}

class Retriever:
    def __init__(self, chunks, tag):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        self.bm25 = BM25Okapi([tok(t) for t in texts])
        emb = embed_cached(texts, tag)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def _dense(self, qvec, n):
        _, ids = self.index.search(np.asarray(qvec, dtype="float32").reshape(1, -1), n)
        return [int(i) for i in ids[0] if i >= 0]

    def _bm25(self, q, n):
        scores = self.bm25.get_scores(tok(q))
        return [int(i) for i in np.argsort(scores)[::-1][:n] if scores[i] > 0]

    def search(self, q, qvec, k=K, mode="hybrid", pool=30):
        if mode == "dense":
            ids = self._dense(qvec, k)
        elif mode == "bm25":
            ids = self._bm25(q, k)
        else:                                         # hybrid = reciprocal rank fusion of both rankings
            fused = defaultdict(float)
            for ranking in (self._dense(qvec, pool), self._bm25(q, pool)):
                for rank, i in enumerate(ranking):
                    fused[i] += 1.0 / (60 + rank + 1)
            ids = sorted(fused, key=fused.get, reverse=True)[:k]
        return [self.chunks[i] for i in ids]

RETRIEVERS = {}
for name, fn in CHUNKERS.items():
    chunks = fn(docs)
    RETRIEVERS[name] = Retriever(chunks, name)
    print(f"{name}: {len(chunks)} chunks indexed")

## 4. Retrieval evaluation (all 1,000 questions, no LLM calls)

Gold = the question's own PubMed ID. `hit@k` = a top-k chunk comes from the gold abstract. `mrr` = mean reciprocal rank of the first such chunk.

In [ ]:
def eval_retrieval(retriever, mode, k=K):
    t0 = time.perf_counter()
    ranks = []
    for i, r in enumerate(questions.itertuples()):
        hits = retriever.search(r.question, Q_EMB[i], k=k, mode=mode)
        ids = [h["doc_id"] for h in hits]
        ranks.append(ids.index(r.pubid) + 1 if r.pubid in ids else None)
    return {"hit@k": float(np.mean([x is not None for x in ranks])),
            "mrr": float(np.mean([1 / x if x else 0.0 for x in ranks])),
            "ms_per_query": (time.perf_counter() - t0) / len(ranks) * 1000}

rows = []
for name, retr in RETRIEVERS.items():
    for mode in ("bm25", "dense", "hybrid"):
        rows.append({"chunker": name, "mode": mode, "n_chunks": len(retr.chunks), **eval_retrieval(retr, mode)})

retrieval_grid = pd.DataFrame(rows).sort_values(["mrr", "hit@k"], ascending=False).reset_index(drop=True)
retrieval_grid.to_csv(RESULTS_DIR / "retrieval_grid.csv", index=False)
display(retrieval_grid.round(3))

## 5. The local LLM

`Qwen3-4B-Instruct-2507` loads in fp16 (the T4 has no bf16). The first run downloads about 8 GB, which takes a few minutes.
If you see an out-of-memory error, set `LOAD_4BIT = True` in the config cell and re-run from there. If the sanity check below prints gibberish or errors about `inf`/`nan`, do the same.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)

def load_llm():
    kw = {"device_map": "auto"}
    if LOAD_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16)
    last = None
    for dtype_arg in ("dtype", "torch_dtype"):        # this argument was renamed across transformers versions
        try:
            return AutoModelForCausalLM.from_pretrained(LLM_ID, **{dtype_arg: torch.float16}, **kw).eval()
        except TypeError as e:
            last = e
    raise last

llm = load_llm()

@torch.inference_mode()
def llm_chat(system, user):
    """Local generation, greedy. Returns (text, prompt_tokens, completion_tokens)."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    n_in = enc["input_ids"].shape[1]
    gen = out[0][n_in:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip(), int(n_in), int(gen.shape[0])

# sanity check
t0 = time.perf_counter()
print(repr(llm_chat("You are a helpful assistant.", "Reply with exactly: Decision: yes")[0]), f"| {time.perf_counter() - t0:.1f}s")

## 6. Answering: closed-book, RAG, and oracle

Three setups, same prompt style, so the comparison is fair:
- `closed_book`: the LLM sees only the question
- `rag`: the LLM sees the top-k retrieved chunks
- `oracle`: the LLM sees the gold abstract (the ceiling for any retrieval improvement)

Small models are more reliable with a simple line format than with JSON, so the model replies `Decision: ...` / `Sources: ...`. Citation correctness is measured from the `Sources` line.

In [ ]:
FORMAT = "Decision: <yes|no|maybe>\nSources: <doc_id>, <doc_id>" + ("\nReason: <one sentence>" if WITH_REASON else "")
SYSTEM_RAG = (
    "You are a biomedical literature assistant. Answer the research question using ONLY the provided abstracts.\n"
    "Decide: yes, no, or maybe. Use 'maybe' only when the abstracts report mixed or inconclusive evidence.\n"
    "In Sources list the doc_id of every abstract you relied on.\n"
    "Reply in exactly this format and nothing else:\n" + FORMAT + "\n"
    "This tool summarises published research. It does not give medical advice."
)
SYSTEM_CLOSED = (
    "You are a biomedical literature assistant. Answer the research question from your general biomedical knowledge.\n"
    "Decide: yes, no, or maybe. Use 'maybe' only when the evidence is mixed or inconclusive.\n"
    "Reply in exactly this format and nothing else (write 'none' for Sources):\n" + FORMAT + "\n"
    "This tool summarises published research. It does not give medical advice."
)

def format_context(hits):
    by_doc = {}
    for h in hits:
        by_doc.setdefault(h["doc_id"], []).append(h["text"])
    return "\n\n".join(f"[doc_id: {d}]\n" + "\n...\n".join(ts) for d, ts in by_doc.items())

def parse_answer(raw):
    m = re.search(r"decision\s*:\s*\**\s*(yes|no|maybe)", raw, re.I)
    if m:
        s = re.search(r"sources?\s*:\s*(.*)", raw, re.I)
        r = re.search(r"reason\s*:\s*(.*)", raw, re.I | re.S)
        return m.group(1).lower(), (re.findall(r"\d+", s.group(1)) if s else []), (r.group(1).strip() if r else ""), True
    m = re.search(r"\b(yes|no|maybe)\b", raw.lower())      # fallback: counts as a parse failure
    return (m.group(1) if m else "invalid"), [], raw[:200], False

def answer(row, setup, retriever=None, search_mode="hybrid", k=K):
    t0 = time.perf_counter()
    retrieved, gold_retrieved = [], None
    if setup == "closed_book":
        system, user = SYSTEM_CLOSED, f"QUESTION: {row.question}"
    elif setup == "oracle":
        system, user = SYSTEM_RAG, f"ABSTRACTS:\n[doc_id: {row.pubid}]\n{row.abstract}\n\nQUESTION: {row.question}"
        retrieved = [row.pubid]
    else:
        hits = retriever.search(row.question, Q_EMB[QIDX[row.pubid]], k=k, mode=search_mode)
        retrieved = list(dict.fromkeys(h["doc_id"] for h in hits))
        gold_retrieved = row.pubid in retrieved
        system, user = SYSTEM_RAG, f"ABSTRACTS:\n{format_context(hits)}\n\nQUESTION: {row.question}"
    raw, tin, tout = llm_chat(system, user)
    decision, cited, why, ok = parse_answer(raw)
    latency = time.perf_counter() - t0
    return {"decision": decision, "rationale": why, "cited": cited, "parsed_ok": ok,
            "retrieved": retrieved, "gold_retrieved": gold_retrieved,
            "cited_gold": (row.pubid in cited) if setup != "closed_book" else None,
            "latency_s": latency, "tokens_in": tin, "tokens_out": tout,
            "cost_usd": latency / 3600 * GPU_USD_PER_HOUR}

# one live test before running the whole eval
test_row = next(dev_df.head(1).itertuples())
out = answer(test_row, "rag", RETRIEVERS["rcts_1000"], "hybrid")
print("gold:", test_row.decision, "| model:", out["decision"], "| cited:", out["cited"], "| gold retrieved:", out["gold_retrieved"])
print(f"{out['latency_s']:.1f}s | {out['tokens_in']}+{out['tokens_out']} tokens")

## 7. Evaluation runner

Each answer is appended to a JSONL file on Drive as it is produced, so if Colab disconnects you simply re-run the cell and it **resumes**. The cache filename includes a hash of the prompts, model and k, so changing any of them starts a fresh run instead of silently reusing stale answers.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def run_hash():
    return hashlib.md5((SYSTEM_RAG + SYSTEM_CLOSED + LLM_ID + str(K) + str(LOAD_4BIT) + str(MAX_NEW)).encode()).hexdigest()[:6]

def summarize(res, label):
    y, p = res["gold"], res["decision"]
    maybe = y == "maybe"
    return {
        "label": label, "n": len(res),
        "accuracy": accuracy_score(y, p),
        "macro_f1": f1_score(y, p, labels=["yes", "no", "maybe"], average="macro", zero_division=0),
        "maybe_recall": float((p[maybe] == "maybe").mean()) if maybe.any() else np.nan,
        "retrieval_hit": pd.to_numeric(res["gold_retrieved"], errors="coerce").mean(),
        "cited_gold": pd.to_numeric(res["cited_gold"], errors="coerce").mean(),
        "parse_fail": int((~res["parsed_ok"].astype(bool)).sum()),
        "p50_latency_s": res["latency_s"].quantile(0.50), "p95_latency_s": res["latency_s"].quantile(0.95),
        "avg_tokens_in": res["tokens_in"].mean(), "avg_cost_usd": res["cost_usd"].mean(),
    }

def run_eval(df, setup, label, retriever=None, search_mode="hybrid", k=K):
    path = RESULTS_DIR / f"gen_{label}_{run_hash()}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            rec = json.loads(line)
            done[rec["pubid"]] = rec
    rows = []
    with path.open("a", encoding="utf-8") as f:
        for row in tqdm(list(df.itertuples()), desc=label):
            if row.pubid in done:
                rows.append(done[row.pubid])
                continue
            rec = answer(row, setup, retriever, search_mode, k)
            rec.update({"pubid": row.pubid, "gold": row.decision, "question": row.question})
            f.write(json.dumps(rec) + "\n")
            f.flush()
            rows.append(rec)
    return summarize(pd.DataFrame(rows), label)

## 8. Run: closed-book vs baseline RAG vs best RAG vs oracle

- Start with `N_RUN = 30` as a smoke test (a few minutes). Then set `N_RUN = None` for the full 150-question dev split.
- **Baseline** = what most tutorials stop at: ~1000-character chunks, dense search only.
- **Best** = the top row of the retrieval grid.
- Keep `EVAL_DF = dev_df` while you iterate. Switch to `test_df` **once**, at the very end, and report that number.

In [ ]:
EVAL_DF, SPLIT = dev_df, "dev"
N_RUN = 30

df_run = EVAL_DF.head(N_RUN) if N_RUN else EVAL_DF
best = retrieval_grid.iloc[0]
print("best retrieval config:", best.chunker, best["mode"], f"(hit@{K}={best['hit@k']:.3f}, mrr={best['mrr']:.3f})")

summaries = [
    run_eval(df_run, "closed_book", f"{SPLIT}_closed_book"),
    run_eval(df_run, "rag", f"{SPLIT}_baseline_rcts1000_dense", RETRIEVERS["rcts_1000"], "dense"),
    run_eval(df_run, "rag", f"{SPLIT}_best_{best.chunker}_{best['mode']}", RETRIEVERS[best.chunker], best["mode"]),
    run_eval(df_run, "oracle", f"{SPLIT}_oracle"),
]
summary = pd.DataFrame(summaries)
summary.loc[len(summary)] = {"label": "majority_class_yes", "n": len(df_run), "accuracy": (df_run.decision == "yes").mean()}
summary.to_csv(RESULTS_DIR / f"summary_{SPLIT}.csv", index=False)
display(summary.round(3))

## What to do next

1. **Read the failures.** Open the `gen_*.jsonl` files and read ~20 wrong answers. Sort each into: retrieval miss, model ignored the evidence, label noise, or "maybe" confusion. "maybe" is typically the hardest class, so look at `maybe_recall`.
2. **Interpret the three-way comparison.** `oracle - rag` is the accuracy you could still gain from better retrieval. `rag - closed_book` is what retrieval is worth. If `oracle` is barely above `rag`, stop tuning retrieval and look at the prompt and the model instead.
3. **Make retrieval harder.** Raise `N_DISTRACTORS` to 20,000 and re-run the grid. Real systems search a big pool, and the gaps between methods get clearer.
4. **Final numbers.** When you are done tuning, set `EVAL_DF, SPLIT = test_df, "test"` and `N_RUN = None`, run once, and use *those* numbers in your README.
5. Push the notebook and `results/` to GitHub (File -> Save a copy in GitHub).

**Notebook 2:** cross-encoder reranking (`BAAI/bge-reranker-base`), reason-first prompting vs answer-first, a larger model in 4-bit vs this one (accuracy vs latency vs memory), and failure-mode analysis.
**Notebook 3:** FastAPI service wrapping the pipeline, Dockerfile, GitHub Actions, a Streamlit or React demo with citations, request logging, and a LangChain/LangGraph agent wrapper.